# ЛР 1_1 - Первичное исследование и оценка качества данных (продолжение)
Датасет: UCI Bike Sharing (hour.csv)

Цель: шкалы измерений, пропуски (MCAR/MAR/MNAR), индикаторы пропусков, выбросы, статистики и визуализации.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

In [ ]:
PATH = "../data/raw/hour.csv"
df = pd.read_csv(PATH)
df.head()

In [ ]:
df["dteday"] = pd.to_datetime(df["dteday"], errors="coerce")

cat_cols = ["season","yr","mnth","hr","holiday","weekday","workingday","weathersit"]
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype("category")

df["datetime"] = df["dteday"] + pd.to_timedelta(df["hr"].astype(int), unit="h")
df.dtypes

## 1. Типы признаков и шкалы измерения

### Задание 1.1
Укажем тип шкалы измерения для признаков датасета.

In [ ]:
scales = {
    "instant": "номинальная (ID)",
    "dteday": "интервальная (время)",
    "datetime": "интервальная (время)",
    "season": "номинальная (код категории)",
    "yr": "номинальная/порядковая (код года)",
    "mnth": "порядковая (месяц, цикличная)",
    "hr": "порядковая (час, цикличная)",
    "holiday": "номинальная (бинарная)",
    "weekday": "порядковая (день недели, цикличная)",
    "workingday": "номинальная (бинарная)",
    "weathersit": "порядковая (условно по коду) / категориальная",
    "temp": "отношений (нормированное число)",
    "atemp": "отношений (нормированное число)",
    "hum": "отношений (нормированное число)",
    "windspeed": "отношений (нормированное число)",
    "casual": "отношений (счетчик)",
    "registered": "отношений (счетчик)",
    "cnt": "отношений (счетчик)"
}

pd.DataFrame({
    "feature": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "scale": [scales.get(c, "не задано") for c in df.columns]
})

**Вывод:**  
Коды категорий (season, weathersit, weekday, hr и др.) записаны числами, но не являются “количественными” в физическом смысле.  
Счетчики спроса (casual, registered, cnt) - шкала отношений: корректны средние/медианы/квантили.  
Погодные признаки количественные, но нормированы (обычно 0..1), поэтому их надо интерпретировать как нормированные величины.

### Задание 1.2
Выберем два признака с разными шкалами: `weathersit` (категория) и `cnt` (счетчик).  
Ответим, имеет ли смысл считать mean/median/mode и что методологически неверно.

In [ ]:
# weathersit: частоты (мода и распределение)
display(df["weathersit"].value_counts(dropna=False))

# cnt: mean/median
df["cnt"].agg(["mean","median","min","max"])

**weathersit (категориальный код):**
- mean/median pandas посчитает, но методологически это неверно, так как код категории не задает осмысленную числовую шкалу.
- мода и частоты корректны.

**cnt (счетчик, шкала отношений):**
- mean и median корректны математически.
- при асимметрии и выбросах median лучше отражает “типичный” спрос, чем mean.

## 2. Пропуски и их природа (MCAR / MAR / MNAR)

В исходном датасете пропусков нет (NaN = 0%), поэтому для демонстрации диагностики создадим искусственные пропуски (MAR-сценарий) в признаке `hum`.
Также текстом сформулируем гипотезы MCAR и MNAR.

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df_mar = df.copy()
rng = np.random.default_rng(42)

# hum пропадает чаще при плохой погоде (weathersit>=3)
mask_mar = (df_mar["weathersit"].astype(int) >= 3) & (rng.random(len(df_mar)) < 0.30)
df_mar.loc[mask_mar, "hum"] = np.nan

df_mar["hum"].isna().mean()

In [ ]:
tmp = df_mar.copy()
tmp["hum_missing"] = tmp["hum"].isna().astype(int)

tmp.groupby("weathersit")["hum_missing"].mean()

**MCAR-гипотеза:** пропуски полностью случайны, не связаны с hr/weathersit/workingday/cnt. Проверка: доли пропусков одинаковы в группах.

**MAR-гипотеза (демонстрация):** пропуски зависят от наблюдаемого признака (например, при weathersit>=3). В проверке видно, что доля пропусков различается по weathersit, значит сценарий соответствует MAR.

**MNAR-гипотеза:** пропуски зависят от истинного значения hum (например, пропадает чаще при высокой влажности). Доказать напрямую сложно, потому что значения скрыты; обычно используют доменные гипотезы и косвенные проверки.

## 3. Индикатор пропуска как источник информации (Задание 3)

Сделаем индикатор пропуска `hum_missing`. Проверим, несет ли он информацию (например, связан ли с cnt).

In [ ]:
tmp.groupby("hum_missing")["cnt"].agg(["mean","median","count"])

1) Факт пропуска несет информацию, когда пропуски систематические (например, зависят от weathersit или других условий).

2) Если индикатор пропуска значим в модели, это часто сигнал, что:
- пропуски не случайны
- и/или импутация выполнена грубо (например, средним), и модель использует индикатор как замену потерянной информации.

3) Пример:
- полезен: MAR (пропуски зависят от weathersit/workingday)
- бесполезен: MCAR (полностью случайно)
- опасен: если пропуски связаны с таргетом cnt (утечка) или MNAR, когда механизм пропуска связан со скрытым значением

## 4. Выбросы (Задание 4)
Выберем признак `cnt` и один выброс по методу IQR. Обсудим: ошибка или сигнал, и что делать.

In [ ]:
x = df["cnt"].dropna()
q1, q3 = x.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr

out = df[df["cnt"] > high].sort_values("cnt", ascending=False)
print("IQR bounds:", (low, high))
out[["datetime","hr","workingday","weathersit","temp","hum","cnt"]].head(1)

Выброс по cnt может быть реальным пиком спроса (час пик, хороший сезон/погода, событие).
Удалять выброс автоматически рискованно - можно потерять важные “пиковые” часы.
Оставлять без изменений безопасно для описательного анализа, но для моделей выбросы могут сильно влиять на среднее и ошибки.
Компромисс: применять log1p/sqrt, робастные модели или анализировать пики отдельно.

## 5. Статистики (Задания 5 и 6)
Рассмотрим асимметричный признак `cnt`: преобразования и сравнение mean/median/geometric mean.

In [ ]:
df_stat = df.copy()
df_stat["cnt_log1p"] = np.log1p(df_stat["cnt"])
p99 = df_stat["cnt"].quantile(0.99)
df_stat["cnt_winsor99"] = df_stat["cnt"].clip(upper=p99)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_stat["cnt"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("cnt"); axes[0].set_xlabel("cnt"); axes[0].set_ylabel("count")

sns.histplot(df_stat["cnt_log1p"], bins=40, kde=True, ax=axes[1])
axes[1].set_title("log1p(cnt)"); axes[1].set_xlabel("log1p(cnt)"); axes[1].set_ylabel("count")
plt.show()

- Логарифмирование (log1p) лучше при сильной правой асимметрии и выбросах: уменьшает “хвост” и делает данные более удобными для линейных моделей, но ухудшает прямую интерпретацию.
- Винзоризация полезна, когда выбросы считаются шумом: она сохраняет шкалу cnt, но ограничивает влияние экстремумов.
На визуализациях log1p обычно “сжимает” хвост сильнее.

In [ ]:
mean_cnt = df["cnt"].mean()
median_cnt = df["cnt"].median()
geo_cnt = np.expm1(np.log1p(df["cnt"]).mean())

mean_cnt, median_cnt, geo_cnt

Среднее, медиана и геометрическое среднее различаются из-за асимметрии распределения cnt и редких больших значений.
- mean чувствительно к хвосту и выбросам
- median устойчивее и чаще лучше отражает “типичный” спрос
- геометрическое среднее полезно при лог-нормальности/относительных изменениях

## 6. Визуализация как инструмент мышления (Задания 7 и 8)
Пара признаков: hr и cnt.

In [ ]:
hour_mean = df.groupby(df["hr"].astype(int))["cnt"].mean()

plt.figure(figsize=(8,8))
plt.pie(hour_mean.values, labels=hour_mean.index)
plt.title("ПЛОХОЙ пример: pie chart среднего cnt по часам")
plt.show()

Pie chart для 24 часов - плохой выбор:
- много категорий, визуально трудно сравнивать
- создается ложная интерпретация “долей”, хотя это средние значения
- не видно суточного профиля (пики/провалы)
Лучше: lineplot или boxplot.

In [ ]:
plt.figure(figsize=(10,4))
sns.lineplot(data=df, x=df["hr"].astype(int), y="cnt", hue="workingday", estimator="mean")
plt.title("Средний cnt по часам: workingday=0/1")
plt.xlabel("hr"); plt.ylabel("mean(cnt)")
plt.show()

plt.figure(figsize=(12,4))
sns.boxplot(data=df, x=df["hr"].astype(int), y="cnt")
plt.title("cnt по часам (boxplot)")
plt.xlabel("hr"); plt.ylabel("cnt")
plt.show()

- lineplot подчеркивает форму суточного профиля и различия рабочий/выходной (пики спроса).
- boxplot подчеркивает разброс и выбросы в каждом часу.
Гипотезы: в рабочие дни выражены пики в часы поездок на работу/с работы; в выходные профиль более ровный.